<div style="text-align: center;">
    <img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*1k72mg1_CZvLptX77zzKTg.png" width="8500"/></div>



# 🕵️‍♀️⏰ **DELAYS... WHO ARE THE HIDDEN CULPRITS?!** ⏰🕵️‍♂️

## **What does Olist do? ?**  
A platform that enables sellers to list and sell their products across multiple major Brazilian marketplaces, providing centralized management and logistics support. A service that manages seller accounts across sales channels, optimizing product listings and pricing for marketplace fulfillment 


# 1- Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from math import radians, sin, cos, sqrt, atan2

from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import learning_curve
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time
from urllib.error import URLError, HTTPError
from socket import timeout


# 2- Loading Data + Merge

In [3]:
df1=pd.read_csv("olist_orders_dataset.csv")
df2= pd.read_csv("olist_order_items_dataset.csv")
df3= pd.read_csv("olist_customers_dataset.csv")
df4=pd.read_csv("olist_order_items_dataset.csv")
df5=pd.read_csv("olist_order_reviews_dataset.csv")
df6=pd.read_csv("olist_products_dataset.csv")
df7=pd.read_csv("product_category_name_translation.csv")
df8=pd.read_csv("olist_sellers_dataset.csv")
df9=pd.read_csv("olist_geolocation_dataset.csv")
df10= pd.read_csv("olist_closed_deals_dataset.csv")
df11=pd.read_csv("olist_marketing_qualified_leads_dataset.csv")

In [4]:
df = pd.merge(df1, df3, on='customer_id') 
df = pd.merge(df, df2, on='order_id') 
df = pd.merge(df, df6, on='product_id')   
df = pd.merge(df, df7, on='product_category_name')    
df = pd.merge(df, df8, on='seller_id')              
df = pd.merge(df, df5, on='order_id')                  
geo = df9.groupby('geolocation_zip_code_prefix').first().reset_index()
df = pd.merge(df, geo, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix')
df = pd.merge(df, geo, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', 
              suffixes=('_customer', '_seller'))

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 46 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   order_id                              110196 non-null  object 
 1   customer_id                           110196 non-null  object 
 2   order_status                          110196 non-null  object 
 3   order_purchase_timestamp              110196 non-null  object 
 4   order_approved_at                     110182 non-null  object 
 5   order_delivered_carrier_date          109061 non-null  object 
 6   order_delivered_customer_date         107919 non-null  object 
 7   order_estimated_delivery_date         110196 non-null  object 
 8   customer_unique_id                    110196 non-null  object 
 9   customer_zip_code_prefix              110196 non-null  int64  
 10  customer_city                         110196 non-null  object 
 11  

In [6]:
columns_to_keep = [
   'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'order_delivered_carrier_date',
    'order_approved_at',
    'product_id',
    'product_category_name_english',
    'seller_id',
    'seller_city',
    'seller_state',
    'seller_zip_code_prefix',
    'customer_id',
    'customer_city',
    'customer_state',
    'customer_zip_code_prefix',
    'review_score',
    'price',
    'freight_value',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'geolocation_lng_customer',
    'geolocation_lng_seller',
    'geolocation_lat_seller',
    'geolocation_lat_customer'
]

df_selected = df[columns_to_keep]
print("Filtered dataset shape:",df_selected.shape)

Filtered dataset shape: (110196, 26)


In [7]:
df_selected.head()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_delivered_carrier_date,order_approved_at,product_id,product_category_name_english,seller_id,seller_city,seller_state,...,price,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm,geolocation_lng_customer,geolocation_lng_seller,geolocation_lat_seller,geolocation_lat_customer
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,2017-10-04 19:55:00,2017-10-02 11:07:15,87285b34884572647811a353c7ac498a,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP,...,29.99,8.72,500.0,19.0,8.0,13.0,-46.587471,-46.452454,-23.680114,-23.574809
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,2018-07-26 14:31:00,2018-07-26 03:24:27,595fac2a385ac33a80bd5114aec74eb8,perfumery,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,SP,...,118.70,22.76,400.0,19.0,13.0,19.0,-44.988369,-43.984727,-19.810119,-12.169860
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,2018-08-08 13:50:00,2018-08-08 08:55:23,aa4383b373c6aca5d8797843e5594415,auto,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,...,159.90,19.22,420.0,24.0,19.0,21.0,-48.514624,-48.232976,-21.362358,-16.746337
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,2017-11-22 13:39:59,2017-11-18 19:45:59,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,66922902710d126a0e7d26b0e3805106,belo horizonte,MG,...,45.00,27.20,450.0,30.0,10.0,20.0,-35.275467,-43.923299,-19.840168,-5.767733
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,2018-02-14 19:46:34,2018-02-13 22:20:29,65266b2da20d04dbe00c5c2d3bb7859e,stationery,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,SP,...,19.90,8.72,250.0,51.0,15.0,15.0,-46.524784,-46.260979,-23.551707,-23.675037


# 3- Data Exploration (EDA) 

In [9]:
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_purchase_timestamp       110196 non-null  object 
 1   order_delivered_customer_date  107919 non-null  object 
 2   order_estimated_delivery_date  110196 non-null  object 
 3   order_delivered_carrier_date   109061 non-null  object 
 4   order_approved_at              110182 non-null  object 
 5   product_id                     110196 non-null  object 
 6   product_category_name_english  110196 non-null  object 
 7   seller_id                      110196 non-null  object 
 8   seller_city                    110196 non-null  object 
 9   seller_state                   110196 non-null  object 
 10  seller_zip_code_prefix         110196 non-null  int64  
 11  customer_id                    110196 non-null  object 
 12  customer_city                 

In [10]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

for col in date_columns:
    df_selected[col] = pd.to_datetime(df_selected[col])
print(df_selected[date_columns].head())

print(df_selected[date_columns].dtypes)

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\168338227.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = pd.to_datetime(df_selected[col])
C:\Users\97333\AppData\Local\Temp\ipykernel_13656\168338227.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = pd.to_datetime(df_selected[col])
C:\Users\97333\AppData\Local\Temp\ipykernel_13656\168338227.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

  order_purchase_timestamp   order_approved_at order_delivered_carrier_date  \
0      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00   
1      2018-07-24 20:41:37 2018-07-26 03:24:27          2018-07-26 14:31:00   
2      2018-08-08 08:38:49 2018-08-08 08:55:23          2018-08-08 13:50:00   
3      2017-11-18 19:28:06 2017-11-18 19:45:59          2017-11-22 13:39:59   
4      2018-02-13 21:18:39 2018-02-13 22:20:29          2018-02-14 19:46:34   

  order_delivered_customer_date order_estimated_delivery_date  
0           2017-10-10 21:25:13                    2017-10-18  
1           2018-08-07 15:27:45                    2018-08-13  
2           2018-08-17 18:06:29                    2018-09-04  
3           2017-12-02 00:28:42                    2017-12-15  
4           2018-02-16 18:17:02                    2018-02-26  
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[n

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\168338227.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = pd.to_datetime(df_selected[col])
C:\Users\97333\AppData\Local\Temp\ipykernel_13656\168338227.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = pd.to_datetime(df_selected[col])


In [11]:
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_purchase_timestamp       110196 non-null  datetime64[ns]
 1   order_delivered_customer_date  107919 non-null  datetime64[ns]
 2   order_estimated_delivery_date  110196 non-null  datetime64[ns]
 3   order_delivered_carrier_date   109061 non-null  datetime64[ns]
 4   order_approved_at              110182 non-null  datetime64[ns]
 5   product_id                     110196 non-null  object        
 6   product_category_name_english  110196 non-null  object        
 7   seller_id                      110196 non-null  object        
 8   seller_city                    110196 non-null  object        
 9   seller_state                   110196 non-null  object        
 10  seller_zip_code_prefix         110196 non-null  int64         
 11  

In [12]:
df_selected.dropna(inplace=True)

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\4136076728.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected.dropna(inplace=True)


In [13]:
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 107903 entries, 0 to 110195
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_purchase_timestamp       107903 non-null  datetime64[ns]
 1   order_delivered_customer_date  107903 non-null  datetime64[ns]
 2   order_estimated_delivery_date  107903 non-null  datetime64[ns]
 3   order_delivered_carrier_date   107903 non-null  datetime64[ns]
 4   order_approved_at              107903 non-null  datetime64[ns]
 5   product_id                     107903 non-null  object        
 6   product_category_name_english  107903 non-null  object        
 7   seller_id                      107903 non-null  object        
 8   seller_city                    107903 non-null  object        
 9   seller_state                   107903 non-null  object        
 10  seller_zip_code_prefix         107903 non-null  int64         
 11  custo

In [14]:
df_selected.duplicated().sum()

10106

In [15]:
df_selected.drop_duplicates()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_delivered_carrier_date,order_approved_at,product_id,product_category_name_english,seller_id,seller_city,seller_state,...,price,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm,geolocation_lng_customer,geolocation_lng_seller,geolocation_lat_seller,geolocation_lat_customer
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,2017-10-04 19:55:00,2017-10-02 11:07:15,87285b34884572647811a353c7ac498a,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP,...,29.99,8.72,500.0,19.0,8.0,13.0,-46.587471,-46.452454,-23.680114,-23.574809
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,2018-07-26 14:31:00,2018-07-26 03:24:27,595fac2a385ac33a80bd5114aec74eb8,perfumery,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,SP,...,118.70,22.76,400.0,19.0,13.0,19.0,-44.988369,-43.984727,-19.810119,-12.169860
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,2018-08-08 13:50:00,2018-08-08 08:55:23,aa4383b373c6aca5d8797843e5594415,auto,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,...,159.90,19.22,420.0,24.0,19.0,21.0,-48.514624,-48.232976,-21.362358,-16.746337
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,2017-11-22 13:39:59,2017-11-18 19:45:59,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,66922902710d126a0e7d26b0e3805106,belo horizonte,MG,...,45.00,27.20,450.0,30.0,10.0,20.0,-35.275467,-43.923299,-19.840168,-5.767733
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2018-02-14 19:46:34,2018-02-13 22:20:29,65266b2da20d04dbe00c5c2d3bb7859e,stationery,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,SP,...,19.90,8.72,250.0,51.0,15.0,15.0,-46.524784,-46.260979,-23.551707,-23.675037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110190,2017-03-09 09:54:05,2017-03-17 15:08:01,2017-03-28,2017-03-10 11:18:03,2017-03-09 09:54:05,ac35486adb7b02598c182c2ff2e05254,health_beauty,e24fc9fcd865784fb25705606fe3dfe7,braganca paulista,SP,...,72.00,13.08,1175.0,22.0,13.0,18.0,-45.889711,-46.524886,-22.957505,-23.178732
110191,2018-02-06 12:58:58,2018-02-28 17:37:56,2018-03-02,2018-02-07 23:22:42,2018-02-06 13:10:37,f1d4ce8c6dd66c47bbaa8c6781c2a923,baby,1f9ab4708f3056ede07124aad39a2554,tupa,SP,...,174.90,20.10,4950.0,40.0,10.0,40.0,-46.446355,-50.497562,-21.935321,-24.001467
110192,2017-08-27 14:46:43,2017-09-21 11:24:17,2017-09-27,2017-08-28 20:52:26,2017-08-27 15:04:16,b80910977a37536adeddd63663f916ad,home_appliances_2,d50d79cb34e38265a8649c383dcffd48,sao paulo,SP,...,205.99,65.02,13300.0,32.0,90.0,22.0,-39.370942,-46.448489,-23.551013,-17.891522
110193,2018-01-08 21:28:27,2018-01-25 23:32:54,2018-02-15,2018-01-12 15:35:03,2018-01-08 21:36:21,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,a1043bafd471dff536d0c462352beb48,ilicinea,MG,...,179.99,40.59,6550.0,20.0,20.0,20.0,-42.690761,-45.827098,-20.944706,-22.555985


#### Adding a new columns

In [17]:
df_selected['Purchased_to_approved']=df_selected['order_purchase_timestamp']-df_selected['order_approved_at']

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\4114863381.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Purchased_to_approved']=df_selected['order_purchase_timestamp']-df_selected['order_approved_at']


In [18]:
df_selected['Approved_to_carrier']=df_selected['order_approved_at']-df_selected['order_delivered_carrier_date']

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\2743773357.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Approved_to_carrier']=df_selected['order_approved_at']-df_selected['order_delivered_carrier_date']


In [20]:
df_selected['Carrier_to_customer']=df_selected['order_delivered_carrier_date']-df_selected['order_delivered_customer_date']

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\638549613.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Carrier_to_customer']=df_selected['order_delivered_carrier_date']-df_selected['order_delivered_customer_date']


In [21]:
# if +ve then late
# if -ve then early
# if 0 then ontime
df_selected['Customer_minus_Estimated']=df_selected['order_delivered_customer_date']-df_selected['order_estimated_delivery_date']

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\309625801.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Customer_minus_Estimated']=df_selected['order_delivered_customer_date']-df_selected['order_estimated_delivery_date']


In [22]:
df_selected['Expected_delivery_Days']=abs(df_selected['order_purchase_timestamp']-df_selected['order_estimated_delivery_date'])

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\2576856104.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Expected_delivery_Days']=abs(df_selected['order_purchase_timestamp']-df_selected['order_estimated_delivery_date'])


In [23]:
df_selected['Actual_Delivery_Days']=abs(df_selected['order_purchase_timestamp']-df_selected['order_delivered_customer_date'])

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\815459108.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Actual_Delivery_Days']=abs(df_selected['order_purchase_timestamp']-df_selected['order_delivered_customer_date'])


In [24]:
df_selected['Diference_Exp_Act']= df_selected['Expected_delivery_Days']-df_selected['Actual_Delivery_Days']

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\2445420063.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Diference_Exp_Act']= df_selected['Expected_delivery_Days']-df_selected['Actual_Delivery_Days']


In [25]:
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 107903 entries, 0 to 110195
Data columns (total 33 columns):
 #   Column                         Non-Null Count   Dtype          
---  ------                         --------------   -----          
 0   order_purchase_timestamp       107903 non-null  datetime64[ns] 
 1   order_delivered_customer_date  107903 non-null  datetime64[ns] 
 2   order_estimated_delivery_date  107903 non-null  datetime64[ns] 
 3   order_delivered_carrier_date   107903 non-null  datetime64[ns] 
 4   order_approved_at              107903 non-null  datetime64[ns] 
 5   product_id                     107903 non-null  object         
 6   product_category_name_english  107903 non-null  object         
 7   seller_id                      107903 non-null  object         
 8   seller_city                    107903 non-null  object         
 9   seller_state                   107903 non-null  object         
 10  seller_zip_code_prefix         107903 non-null  int64        

In [26]:
def categorize_delivery(time_difference):
    zero_timedelta = pd.Timedelta(0, unit='days')

    if time_difference > zero_timedelta:
        return 'late'
    elif time_difference < zero_timedelta:
        return 'early'
    else:
        return 'ontime'
df_selected['delivery_status'] = df_selected['Customer_minus_Estimated'].apply(categorize_delivery)

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\2534537235.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['delivery_status'] = df_selected['Customer_minus_Estimated'].apply(categorize_delivery)


In [27]:
df_selected['delivery_status'].value_counts()

delivery_status
early    99537
late      8366
Name: count, dtype: int64

#### Adding a new column for each month:
##### Dec 12 to March 3 then Summer
##### April 4 to June 6 then Autumn
##### July 7 to September 8 then Winter
##### October 9 to November 11 then Spring

In [29]:
df_selected['Month_Of_Exp_Del']= df_selected['order_estimated_delivery_date'].dt.month

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\2490331534.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Month_Of_Exp_Del']= df_selected['order_estimated_delivery_date'].dt.month


In [30]:
seasons = {
    'Summer': [12, 1, 2, 3],
    'Autumn': [4, 5, 6],
    'Winter': [7, 8, 9],
    'Spring': [10, 11],
}

In [31]:
def get_weather_category(month):
    for season, months in seasons.items():
        if month in months:
            return season
    return 'Undefined' 

In [32]:
df_selected['weather_category'] = df_selected['Month_Of_Exp_Del'].apply(get_weather_category)

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\3949744676.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['weather_category'] = df_selected['Month_Of_Exp_Del'].apply(get_weather_category)


In [33]:
df_selected.head(2)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_delivered_carrier_date,order_approved_at,product_id,product_category_name_english,seller_id,seller_city,seller_state,...,Purchased_to_approved,Approved_to_carrier,Carrier_to_customer,Customer_minus_Estimated,Expected_delivery_Days,Actual_Delivery_Days,Diference_Exp_Act,delivery_status,Month_Of_Exp_Del,weather_category
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,2017-10-04 19:55:00,2017-10-02 11:07:15,87285b34884572647811a353c7ac498a,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP,...,-1 days +23:49:18,-3 days +15:12:15,-7 days +22:29:47,-8 days +21:25:13,15 days 13:03:27,8 days 10:28:40,7 days 02:34:47,early,10,Spring
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,2018-07-26 14:31:00,2018-07-26 03:24:27,595fac2a385ac33a80bd5114aec74eb8,perfumery,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,SP,...,-2 days +17:17:10,-1 days +12:53:27,-13 days +23:03:15,-6 days +15:27:45,19 days 03:18:23,13 days 18:46:08,5 days 08:32:15,early,8,Winter


#### Adding a new column for weather events :

In [35]:
weather_events = {
    'Sao Paulo Floods': (pd.to_datetime('2016-03-01'), pd.to_datetime('2016-03-31')),  # March 2016
    'Santos Port Flooding': (pd.to_datetime('2016-08-01'), pd.to_datetime('2016-08-31')),  # August 2016
    'BR-163 Highway Rains': (pd.to_datetime('2017-12-01'), pd.to_datetime('2017-12-31')),  # December 2017
    'Barcarena Flooding': (pd.to_datetime('2018-02-01'), pd.to_datetime('2018-02-28')),  # February 2018
}

In [36]:
def get_weather_event(date):
    for event, (start, end) in weather_events.items():
        if start <= date <= end:
            return event
    return 'None'  

In [37]:
df_selected['weather_event'] = df_selected['order_estimated_delivery_date'].apply(get_weather_event)

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\378677749.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['weather_event'] = df_selected['order_estimated_delivery_date'].apply(get_weather_event)


In [39]:
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 107903 entries, 0 to 110195
Data columns (total 37 columns):
 #   Column                         Non-Null Count   Dtype          
---  ------                         --------------   -----          
 0   order_purchase_timestamp       107903 non-null  datetime64[ns] 
 1   order_delivered_customer_date  107903 non-null  datetime64[ns] 
 2   order_estimated_delivery_date  107903 non-null  datetime64[ns] 
 3   order_delivered_carrier_date   107903 non-null  datetime64[ns] 
 4   order_approved_at              107903 non-null  datetime64[ns] 
 5   product_id                     107903 non-null  object         
 6   product_category_name_english  107903 non-null  object         
 7   seller_id                      107903 non-null  object         
 8   seller_city                    107903 non-null  object         
 9   seller_state                   107903 non-null  object         
 10  seller_zip_code_prefix         107903 non-null  int64        

#### The distance between the seller and the customer

In [41]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [42]:
df_selected.seller_state.unique()

array(['SP', 'MG', 'ES', 'RS', 'DF', 'PR', 'SC', 'RJ', 'GO', 'BA', 'MA',
       'PB', 'PE', 'CE', 'MT', 'PI', 'RN', 'MS', 'PA', 'AM', 'SE', 'RO'],
      dtype=object)

In [43]:
df_selected.customer_state.unique()

array(['SP', 'BA', 'GO', 'RN', 'PR', 'RJ', 'RS', 'MG', 'SC', 'RR', 'PE',
       'TO', 'CE', 'DF', 'SE', 'MT', 'PB', 'PA', 'RO', 'ES', 'AP', 'MS',
       'MA', 'PI', 'AL', 'AM', 'AC'], dtype=object)

In [44]:
customer_states = ['SP', 'BA', 'GO', 'RN', 'PR', 'RJ', 'RS', 'MG', 'SC', 'RR', 'PE',
                   'TO', 'CE', 'DF', 'SE', 'MT', 'PB', 'PA', 'RO', 'ES', 'AP', 'MS',
                   'MA', 'PI', 'AL', 'AC', 'AM']
seller_states = ['SP', 'MG', 'ES', 'RS', 'DF', 'PR', 'SC', 'RJ', 'GO', 'BA', 'MA',
                 'PB', 'PE', 'CE', 'MT', 'PI', 'RN', 'MS', 'PA', 'AM', 'SE', 'RO']

state_coordinates = {}
geolocator = Nominatim(user_agent="state_distance_calculator")
for state in np.unique(customer_states + seller_states):
    location = geolocator.geocode(state + ", Brazil")
    if location:
        state_coordinates[state] = (location.latitude, location.longitude)
    else:
        print(f"Could not find coordinates for state: {state}")
        state_coordinates[state] = None 

def calculate_distance(customer_state, seller_state):
    coord_customer = state_coordinates.get(customer_state)
    coord_seller = state_coordinates.get(seller_state)

    if coord_customer and coord_seller:
        return geodesic(coord_customer, coord_seller).km
    else:
        return None  

df_selected['distance_km'] = df_selected.apply(
    lambda row: calculate_distance(row['customer_state'], row['seller_state']), axis=1
)
print(df_selected[['customer_state', 'seller_state', 'distance_km']].head())


  customer_state seller_state  distance_km
0             SP           SP     0.000000
1             BA           SP  1284.513296
2             GO           SP   702.734138
3             RN           MG  1648.049807
4             SP           SP     0.000000


C:\Users\97333\AppData\Local\Temp\ipykernel_13656\3699333878.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['distance_km'] = df_selected.apply(


In [79]:
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
Index: 107903 entries, 0 to 110195
Data columns (total 38 columns):
 #   Column                         Non-Null Count   Dtype          
---  ------                         --------------   -----          
 0   order_purchase_timestamp       107903 non-null  datetime64[ns] 
 1   order_delivered_customer_date  107903 non-null  datetime64[ns] 
 2   order_estimated_delivery_date  107903 non-null  datetime64[ns] 
 3   order_delivered_carrier_date   107903 non-null  datetime64[ns] 
 4   order_approved_at              107903 non-null  datetime64[ns] 
 5   product_id                     107903 non-null  object         
 6   product_category_name_english  107903 non-null  object         
 7   seller_id                      107903 non-null  object         
 8   seller_city                    107903 non-null  object         
 9   seller_state                   107903 non-null  object         
 10  seller_zip_code_prefix         107903 non-null  int64        

In [80]:
df_selected['same_state'] = df_selected['seller_state'] == df_selected['customer_state']

C:\Users\97333\AppData\Local\Temp\ipykernel_13656\2734162689.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['same_state'] = df_selected['seller_state'] == df_selected['customer_state']


In [82]:
df_selected['same_state'].value_counts()

same_state
False    68717
True     39186
Name: count, dtype: int64

####  Load weather data

In [90]:
import pandas as pd

def load_weather_data_from_csv(csv_file_path):
    try:
        df = pd.read_csv(csv_file_path)
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
        df = df.dropna(subset=['datetime'])  
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {csv_file_path}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while loading the CSV file: {e}")
        return None

def get_weather_on_delivery_date(delivery_date, weather_df):
    if weather_df is None:
        return "No weather data available"

    if isinstance(delivery_date, pd.Timestamp):
        delivery_date = delivery_date.date()

    weather_row = weather_df[weather_df['datetime'].dt.date == delivery_date]

    if not weather_row.empty:

        return f"Weather on {delivery_date}: {weather_row['conditions'].values[0]}, Temp: {weather_row['temp'].values[0]}C, Wind: {weather_row['windspeed'].values[0]} km/h"
    else:
        return 'No data found'

csv_file_path = 'Brazil 2016-10-01 to 2018-12-31.csv' 
weather_df = load_weather_data_from_csv(csv_file_path)

df = df_selected.copy()

if weather_df is not None:
    df['weather_on_delivery_date'] = df['order_estimated_delivery_date'].apply(
        lambda date: get_weather_on_delivery_date(date, weather_df))
    print(df[['order_estimated_delivery_date', 'weather_on_delivery_date']].head())
else:
    print("Failed to retrieve weather data.  The 'weather_on_delivery_date' column was not added.")


  order_estimated_delivery_date  \
0                    2017-10-18   
1                    2018-08-13   
2                    2018-09-04   
3                    2017-12-15   
4                    2018-02-26   

                            weather_on_delivery_date  
0  Weather on 2017-10-18: Clear, Temp: 76.4C, Win...  
1  Weather on 2018-08-13: Clear, Temp: 71.4C, Win...  
2  Weather on 2018-09-04: Clear, Temp: 70.1C, Win...  
3  Weather on 2017-12-15: Rain, Partially cloudy,...  
4  Weather on 2018-02-26: Rain, Partially cloudy,...  


In [91]:
weather_df

,name,datetime,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,solarenergy,uvindex,severerisk,sunrise,sunset,moonphase,conditions,description,icon,stations
0,Brazil,2016-10-01,81.6,66.3,73.7,81.1,66.3,73.7,58.7,60.7,...,17.5,7.0,NaN,2016-10-01T05:53:52,2016-10-01T18:08:46,0.02,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"86715099999,86714099999,SBBR,83378099999,86716..."
1,Brazil,2016-10-02,86.1,66.0,75.3,84.2,66.0,75.1,58.6,58.7,...,24.3,9.0,NaN,2016-10-02T05:53:07,2016-10-02T18:08:53,0.05,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"86715099999,86714099999,SBBR,83378099999,86716..."
2,Brazil,2016-10-03,82.5,69.2,74.9,82.2,69.2,74.8,60.1,61.6,...,22.5,10.0,NaN,2016-10-03T05:52:21,2016-10-03T18:09:01,0.09,"Rain, Partially cloudy",Partly cloudy throughout the day with afternoo...,rain,"86715099999,86714099999,SBBR,83378099999,86716..."
3,Brazil,2016-10-04,71.2,63.8,66.9,71.2,63.8,66.9,62.4,85.7,...,10.6,4.0,NaN,2016-10-04T05:51:37,2016-10-04T18:09:09,0.12,"Rain, Partially cloudy",Partly cloudy throughout the day with rain in ...,rain,"86715099999,86714099999,SBBR,83378099999,86716..."
4,Brazil,2016-10-05,76.3,62.0,66.6,76.3,62.0,66.6,61.3,83.7,...,9.6,5.0,NaN,2016-10-05T05:50:52,2016-10-05T18:09:17,0.15,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,"86715099999,86714099999,SBBR,83378099999,86716..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,Brazil,2018-12-27,80.9,65.6,70.8,82.2,65.6,70.9,63.8,80.4,...,19.4,7.0,NaN,2018-12-27T06:41:01,2018-12-27T19:44:34,0.67,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,"86715099999,86714099999,SBBR,83378099999,86716..."
818,Brazil,2018-12-28,77.1,65.6,70.0,77.1,65.6,70.0,64.1,82.2,...,14.3,8.0,NaN,2018-12-28T06:41:35,2018-12-28T19:44:59,0.70,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"86715099999,86714099999,SBBR,83378099999,86716..."
819,Brazil,2018-12-29,78.2,62.0,69.8,78.2,62.0,69.8,61.1,75.3,...,12.3,5.0,NaN,2018-12-29T06:42:08,2018-12-29T19:45:24,0.75,"Rain, Partially cloudy",Partly cloudy throughout the day with rain cle...,rain,"86715099999,86714099999,SBBR,83378099999,86716..."
820,Brazil,2018-12-30,80.9,65.6,73.1,81.7,65.6,73.1,62.4,71.0,...,24.2,10.0,NaN,2018-12-30T06:42:42,2018-12-30T19:45:47,0.77,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"86715099999,86714099999,SBBR,83378099999,86716..."


In [95]:
new_delimiter = ','
df[['weather_condition', 'temp']] = df['weather_on_delivery_date'].str.split(new_delimiter, n=1, expand=True)

In [103]:
df.head(2)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_delivered_carrier_date,order_approved_at,product_id,product_category_name_english,seller_id,seller_city,seller_state,...,Diference_Exp_Act,delivery_status,Month_Of_Exp_Del,weather_category,weather_event,distance_km,same_state,weather_on_delivery_date,weather_condition,temp
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,2017-10-04 19:55:00,2017-10-02 11:07:15,87285b34884572647811a353c7ac498a,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP,...,7 days 02:34:47,early,10,Spring,None,0.000000,True,"Weather on 2017-10-18: Clear, Temp: 76.4C, Win...",Weather on 2017-10-18: Clear,"Temp: 76.4C, Wind: 11.7 km/h"
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,2018-07-26 14:31:00,2018-07-26 03:24:27,595fac2a385ac33a80bd5114aec74eb8,perfumery,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,SP,...,5 days 08:32:15,early,8,Winter,None,1284.513296,False,"Weather on 2018-08-13: Clear, Temp: 71.4C, Win...",Weather on 2018-08-13: Clear,"Temp: 71.4C, Wind: 8.2 km/h"


#### The distance between the seller and the customer by the geolocation/zip code

In [96]:
!pip install pgeocode

In [97]:
import pgeocode
nomi = pgeocode.Nominatim('us')
a = nomi.query_postal_code('36067')
print(a['latitude'], a['longitude'])

32.4715 -86.4831


In [105]:
from geopy.distance import geodesic
seller_coords = (df['geolocation_lat_seller'].iloc[0], df['geolocation_lng_seller'].iloc[0])
customer_coords = (df['geolocation_lat_customer'].iloc[0], df['geolocation_lng_customer'].iloc[0])

distance_km = geodesic(seller_coords, customer_coords).km
print(f"Distance: {distance_km} km")

Distance: 18.051106106034556 km


In [112]:
df_selected['distance_km'] = df_selected.apply(
    lambda row: geodesic(
        (row['geolocation_lat_seller'], row['geolocation_lng_seller']),
        (row['geolocation_lat_customer'], row['geolocation_lng_customer'])
    ).km,axis=1)
print(df_selected[['seller_city', 'customer_city', 'distance_km']].head())

       seller_city            customer_city  distance_km
0             maua                sao paulo    18.051106
1   belo horizonte                barreiras   852.256379
2          guariba               vianopolis   511.820721
3   belo horizonte  sao goncalo do amarante  1816.652139
4  mogi das cruzes              santo andre    30.189028


In [113]:
nomi = pgeocode.Nominatim('br')

def get_coordinates(zip_code):
    location = nomi.query_postal_code(str(zip_code).zfill(5))
    return (location['latitude'], location['longitude'])

def calculate_distance(row):
    try:
        seller_coords = get_coordinates(row['seller_zip_code_prefix'])
        customer_coords = get_coordinates(row['customer_zip_code_prefix'])
        
        if None not in seller_coords + customer_coords:
            return geodesic(seller_coords, customer_coords).km
        return None
    except:
        return None

df_selected['distance_from_zip_km'] = df_selec.ted.apply(calculate_distance, axis=1)

if 'distance_km' in df_selected.columns:
    print(df_selected[['distance_km', 'distance_from_zip_km']].head())

   distance_km distance_from_zip_km
0    18.051106                 None
1   852.256379                 None
2   511.820721                 None
3  1816.652139                 None
4    30.189028                 None


#### Where the delay is coming from? ther seller/shipping/ets

In [115]:
df_selected.groupby('same_state')['Carrier_to_customer'].mean()

same_state
False   -12 days +08:07:01.079660055
True     -5 days +06:25:26.824631246
Name: Carrier_to_customer, dtype: timedelta64[ns]

In [116]:
df_selected.groupby('weather_category')['Carrier_to_customer'].mean()

weather_category
Autumn   -10 days +13:31:23.755367031
Spring    -9 days +15:09:03.816190573
Summer   -12 days +17:03:09.229187712
Winter    -7 days +01:13:07.505634435
Name: Carrier_to_customer, dtype: timedelta64[ns]

In [117]:
delay_by_state = df_selected.groupby('customer_state')['Carrier_to_customer'].mean().sort_values(ascending=False)
print(delay_by_state.head())
print(df_selected[['distance_km', 'Carrier_to_customer']].corr())

customer_state
SP    -6 days +11:46:55.331005495
PR    -9 days +10:06:17.237022351
MG    -9 days +07:21:04.209069401
DF   -10 days +07:19:18.898604652
SC   -12 days +12:44:18.117705736
Name: Carrier_to_customer, dtype: timedelta64[ns]
                     distance_km  Carrier_to_customer
distance_km             1.000000            -0.423901
Carrier_to_customer    -0.423901             1.000000


In [ ]:
df_selected['Carrier_to_customer_days'] = df_selected['Carrier_to_customer'].dt.total_seconds() / (60 * 60 * 24)

In [ ]:
df_selected[['product_weight_g', 'Carrier_to_customer']].head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_selected['product_weight_g'], df_selected['Carrier_to_customer_days'], color='teal', alpha=0.6, edgecolors='w', linewidth=0.5)

plt.title('Analyzing the Impact of Product Weight on Delivery Time', fontsize=14, fontweight='bold')
plt.xlabel('Product Weight (grams)', fontsize=12)
plt.ylabel('Delivery Duration (Days)', fontsize=12)

z = np.polyfit(df_selected['product_weight_g'], df_selected['Carrier_to_customer_days'], 1)
p = np.poly1d(z)
plt.plot(df_selected['product_weight_g'], p(df_selected['product_weight_g']), "r--", label='Trendline')

plt.annotate('Slight upward trend\nsuggests heavier items\nmay take longer to deliver.',
             xy=(1600, p(1600)), xytext=(1200, 9),
             arrowprops=dict(facecolor='black', arrowstyle='->'),
             fontsize=10, backgroundcolor='white')

plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
delay_components = ['Purchased_to_approved', 'Approved_to_carrier', 'Carrier_to_customer']
df_plot = df_selected[delay_components].apply(lambda x: x.dt.total_seconds() / 3600) 
sns.boxplot(data=df_plot)
plt.title('Where is the Delivery Time Spent? (in Hours)')
plt.ylabel('Hours')
plt.xticks(rotation=15)
plt.show()


In [ ]:
df_weather = df_selected.groupby('weather_category')['Carrier_to_customer_days'].mean().sort_values()
df_weather.plot(kind='barh', color='skyblue', figsize=(8,5))
plt.title('Average Shipping Delay by Weather Category')
plt.xlabel('Days')
plt.show()

In [ ]:
df['Diference_Exp_Act_days'] = df['Diference_Exp_Act'].dt.total_seconds() / (3600 * 24)
weather_delay = df.groupby('weather_category')['Diference_Exp_Act_days'].mean().sort_values()
distance_corr = df[['distance_km', 'Diference_Exp_Act_days']].corr().iloc[0, 1]
weight_corr = df[['product_weight_g', 'Diference_Exp_Act_days']].corr().iloc[0, 1]
monthly_delay = df.groupby('Month_Of_Exp_Del')['Diference_Exp_Act_days'].mean()
weather_delay.plot(kind='barh', figsize=(8,5), color='cornflowerblue')
plt.title('Average Delivery Delay by Weather Category')
plt.xlabel('Average Delay (Days)')
plt.tight_layout()
plt.show()
monthly_delay.plot(marker='o', linestyle='-', figsize=(8,5), color='seagreen')
plt.title('Average Delivery Delay by Month')
plt.xlabel('Month')
plt.ylabel('Average Delay (Days)')
plt.grid(True)
plt.tight_layout()
plt.show()
print(f"Correlation between Distance and Delay: {distance_corr:.2f}")
print(f"Correlation between Weight and Delay: {weight_corr:.2f}")


In [ ]:
state_delay = df.groupby('customer_state')['Diference_Exp_Act_days'].mean().sort_values(ascending=False)
top_states = state_delay.head(10)
plt.figure(figsize=(10, 6))
top_states.plot(kind='bar', color='tomato')
plt.title('Top 10 Customer States with Highest Average Delivery Delay')
plt.ylabel('Average Delay (Days)')
plt.xlabel('Customer State')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
category_delay = df.groupby('product_category_name_english')['Diference_Exp_Act_days'].mean().sort_values(ascending=False)
top_categories = category_delay.head(10)
plt.figure(figsize=(12, 6))
top_categories.plot(kind='bar', color='skyblue')
plt.title('Top 10 Product Categories with Highest Average Delivery Delay')
plt.ylabel('Average Delay (Days)')
plt.xlabel('Product Category')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
category_delay_count = df.groupby('product_category_name_english').agg(
    avg_delay=('Diference_Exp_Act_days', 'mean'),
    order_count=('order_id', 'count')
).sort_values(by='order_count', ascending=False)
top_categories = category_delay_count.head(10)
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(top_categories.index, top_categories['avg_delay'], color='skyblue', alpha=0.7, label='Average Delay (Days)')
ax1.set_xlabel('Product Category')
ax1.set_ylabel('Average Delay (Days)', color='skyblue')
ax1.tick_params(axis='y', labelcolor='skyblue')
ax2 = ax1.twinx()
ax2.plot(top_categories.index, top_categories['order_count'], color='orange', marker='o', label='Order Count')
ax2.set_ylabel('Order Count', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')
plt.title('Top 10 Product Categories by Order Count and Average Delivery Delay')
ax1.set_xticklabels(top_categories.index, rotation=45)
ax1.grid(axis='y')
plt.tight_layout()
plt.show()


# 3- ML

In [ ]:
df_selected.describe()

In [ ]:
correlation_matrix = df_selected.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of df_selected')
plt.show()

In [ ]:
weather_cols = ['weather_category', 'weather_condition']
existing_weather_cols = [col for col in weather_cols if col in df.columns]

if existing_weather_cols:
    df = pd.get_dummies(df, columns=existing_weather_cols, drop_first=True)
features = [
    'product_weight_g', 'review_score', 'seller_zip_code_prefix',
    'freight_value', 'customer_zip_code_prefix', 'seller_zip_code_prefix.1',
    'Month_Of_Exp_Del', 'distance_km', 'same_state']
features += [col for col in df.columns if col.startswith('weather_category_') or col.startswith('weather_condition_')]
target = 'difference_exp_act_days'
X = df[features]
y = df[[target]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    min_samples_split=5,
    random_state=42
)
model.fit(X_train, y_train.values.ravel())

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("Gradient Boosting Model Performance (Predicting Delivery Delay Days):")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='darkorange')
plt.xlabel('Actual Delivery Delay (days)')
plt.ylabel('Predicted Delivery Delay (days)')
plt.title('Actual vs Predicted Delivery Delay')
plt.plot(y_test, y_test, color='red', linestyle='--')
plt.grid(True)
plt.show()

feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='coolwarm')
plt.title('Feature Importance: Predicting Delivery Delay Days')
plt.tight_layout()
plt.show()
